# Calculate and plot power versus radial uv distance
## For making Figures 14 and 15 in the paper
### A. Ordog, Sept 3, 2024
### based on 'COMBINE_RADIAL_ANALYSIS' series of notebooks from PhD

In [ ]:
import astropy.io.fits as pf
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import matplotlib.pyplot as plt
import math
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
import os
import subprocess
import importlib as imp
import copy
import scipy.stats
from matplotlib.pyplot import cm
from scipy.stats import kde
#import figure_subroutines as subs
from astropy import units as u
from astropy.coordinates import SkyCoord
from pylab import *
from matplotlib import ticker

In [ ]:
def get_fields_and_mosaics(mos_file):
    
    mos_fields = {}
    fields = []
    
    with open(mos_file, 'r') as file:
        for line in file:
            if len(line.strip().split()) == 2:
                mos = line.strip().split()[0]
                mos_fields[mos] = []
            if len(line.strip().split()) == 1:
                field = line.strip().split()[0]
                mos_fields[mos].append(field)
                fields.append(field)
                
    print('Number of mosaics: ',len(mos_fields))
    print('Number of fields in mosaics: ',len(fields))
    
    fields = set(fields)
    print('Number of unique fields: ',len(fields))
    
    return mos_fields, fields

In [ ]:
print('Original CGPS')
mos_fields_old, fieldnames_old = get_fields_and_mosaics('/home/ordoga/DRAO_export/CG_W23/mos_plots/mosaics_fields_list.txt')
print()
print('Actual OB + CGPS')
mos_fields, fieldnames = get_fields_and_mosaics('/home/ordoga/DRAO_export/mosaics_fields_list_OB_and_CGPS.txt')


In [ ]:
diff = list(set(fieldnames) - set(fieldnames_old))

print(diff)

# 384 CGPS
# 1 extra CGPS
# 14 OB (not OB06,OB07,OB01)
# 4 extra archival
print(384+1+14+4)

## Make dictionary for all fields

In [ ]:
#fieldnames = ['ob02','ob03','ob04','ob05',
#              'ob08','ob09','ob10','ob11',
#              'ob12','ob13','ob14','ob15',
#              'ob16','ob17','sn01','sn02',
#              'sn03','rr22']
#fieldnames = ['ej1','ob03','ob04']

fieldnames = list(fieldnames)

numfields = len(fieldnames)
print(numfields)
rmax = 200

data = {}
#channels = {'a':np.zeros(rmax),'b':np.zeros(rmax),'c':np.zeros(rmax),'d':np.zeros(rmax)}
#cgps  = {'raw':channels.copy(), 'feather':channels.copy()}
#gmims = {'raw':channels.copy(),  'deconv':channels.copy(), 'feather':channels.copy()}

for field in fieldnames:
    data[field] = {'stokesq': {'cgps':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                              'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                       'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'gmims':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                               'c':np.zeros(rmax),'d':np.zeros(rmax)},  
                                        'deconv':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                        'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                   'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'ratio':{'a':np.nan,'b':np.nan,'c':np.nan,'d':np.nan}},
                   'stokesu': {'cgps':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                              'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                       'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'gmims':{'raw':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                               'c':np.zeros(rmax),'d':np.zeros(rmax)},  
                                        'deconv':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                  'c':np.zeros(rmax),'d':np.zeros(rmax)}, 
                                        'feather':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                                                   'c':np.zeros(rmax),'d':np.zeros(rmax)}},
                               'ratio':{'a':np.nan,'b':np.nan,'c':np.nan,'d':np.nan}},
                   'uvr2d':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                          'c':np.zeros(rmax),'d':np.zeros(rmax)},
                   'uvr1d':{'a':np.zeros(rmax),'b':np.zeros(rmax),
                          'c':np.zeros(rmax),'d':np.zeros(rmax)}}
    

In [ ]:
def read_data(file):
    
    hdu = fits.open(file)
    hdr = hdu[0].header
    
    n1 = int(hdr['NAXIS1']/2)
    n2 = int(hdr['NAXIS2']-1)
    #print(n1,n2)
    
    data = hdu[0].data[0][0][1:n2+1,0:n1]
    #print(data.shape)


    return data, hdr

In [ ]:
def make_axis_lists_2D(hdr):
    
    nx = hdr['NAXIS1']
    ny = hdr['NAXIS2']
   
    dx = hdr['CDELT1']
    dy = hdr['CDELT2']
    
    xpix = hdr['CRPIX1']
    ypix = hdr['CRPIX2']
    
    xval = hdr['CRVAL1']
    yval = hdr['CRVAL2']
    
    x = np.arange(nx)+1-xpix
    y = np.arange(ny)+1-ypix

    lon_ax = x*dx+xval
    lat_ax = y*dy+yval
    
    return lon_ax, lat_ax

In [ ]:
def get_uv_radii(hdr):

    n1 = int(hdr['NAXIS1']/2)
    n2 = int(hdr['NAXIS2']-1)
    
    ruv = np.zeros([n2,n1])
    
    u,v = make_axis_lists_2D(hdr)
    
    for i in range(0,n1):
        for j in range(0,n2):
            ruv[j,i] = np.sqrt((np.flip(u[1:n1+1])[i])**2 + v[1:n2+1][j]**2)
        
    return ruv

In [ ]:
def plot_uv_radius2D(uvr_field):
    
    
    fig,axs = plt.subplots(1,4,figsize=(12,5))
    for i in range(0,4):
        axs[i].imshow(uvr_field[i],vmin=0,vmax=1000,cmap='gray_r')
        
    fig,ax = plt.subplots(1,1,figsize=(12,5))
    for i in range(0,4):
        idx = int(floor(uvr_field[i].shape[0]/2))    
        ax.plot(uvr_field[i][idx,:])
        
    return
        

In [ ]:
def binning(uvmap,uvr,rmax,r):
    
    binned = np.empty_like(r)
    
    for i in range(0,rmax):
    
        wx = np.where((uvr == r[i]))[1]
        wy = np.where((uvr == r[i]))[0]
        binned[i] = np.nanmean(uvmap[wy,wx])
    
    return binned

## Loop over fields and fill in information

In [ ]:
%%time

#directory = '/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/'
directory = '/home/ordoga/DRAO_export/CG_W23/UV_IMAGES/'

all_fields = True

counter = 1

for field in fieldnames:
    
    print(counter, field)
    
    if all_fields:
    #if field == 'ob12':
        #uvr_sample = []
    
        for stokes in ['q','u']:

            for chan in ['a','b','c','d']:
                
                try:

                    ######################
                    # GMIMS
                    ######################

                    # GMIMS Raw map
                    file = directory+'f5_G_initial_uv_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)  
                    uvr = get_uv_radii(hdr)

                    if stokes == 'q': # only need to record the radii once
                        data[field]['uvr2d'][chan] = uvr
                        data[field]['uvr1d'][chan] = np.unique(uvr)[0:rmax]

                        #if field == fieldnames[1]:
                        #   uvr_sample.append(uvr)

                    data[field]['stokes'+stokes]['gmims']['raw'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # GMIMS deconvolved map
                    file = directory+'f10_G_final_taper_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['gmims']['deconv'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # GMIMS feathered map
                    file = directory+'f11_G_feathered_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['gmims']['feather'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    ######################
                    # CGPS
                    ######################

                    # CGPS raw map
                    file = directory+'f13_C_initial_uv_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['cgps']['raw'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])

                    # CGPS feathered map
                    file = directory+'f14_C_feathered_'+field+'_'+chan+stokes+'.fits'
                    uvmap, hdr = read_data(file)
                    data[field]['stokes'+stokes]['cgps']['feather'][chan] = binning(uvmap,uvr,rmax,data[field]['uvr1d'][chan])


                    w = np.where((data[field]['uvr1d'][chan]>9) & (data[field]['uvr1d'][chan]<17))
                    C_test = data[field]['stokes'+stokes]['cgps']['raw'][chan][w]
                    G_test = data[field]['stokes'+stokes]['gmims']['deconv'][chan][w]
                    #print(C_test.shape,G_test.shape)
                    data[field]['stokes'+stokes]['ratio'][chan] = np.nanmean(G_test/C_test)
                    
                except:
                    print('no file')
                    pass

            
    counter = counter+1


In [ ]:
print(data['ob12']['stokesq']['ratio']['d'])

np.savez('field_info.npz', data=data)

In [ ]:
make_radial_plots(data,'ob12','stokesu','c')


In [ ]:
# Load the .npz file, allowing pickled objects
data = np.load('field_info_v2.npz', allow_pickle=True)

# Access the content under the first key
first_key = data.files[0]
content = data[first_key]

# Check if the content is a pickled dictionary
if isinstance(content, np.ndarray) and content.dtype == 'object':
    fields_data = content.item()  # This should restore the dictionary
    #print(restored_dict)  # Now you can work with the dictionary
else:
    print("Content is not a pickled dictionary.")

In [ ]:
print(fields_data['ob12'])

In [ ]:
field_name = list(fields_data.keys())
#print(field_name)

num = len(fields_data)

ratios_q = np.empty([4,num])
ratios_u = np.empty([4,num])

for i in range(0,num):

    ratios_q[0,i] = fields_data[field_name[i]]['stokesq']['ratio']['a']
    ratios_q[1,i] = fields_data[field_name[i]]['stokesq']['ratio']['b']
    ratios_q[2,i] = fields_data[field_name[i]]['stokesq']['ratio']['c']
    ratios_q[3,i] = fields_data[field_name[i]]['stokesq']['ratio']['d']

    ratios_u[0,i] = fields_data[field_name[i]]['stokesu']['ratio']['a']
    ratios_u[1,i] = fields_data[field_name[i]]['stokesu']['ratio']['b']
    ratios_u[2,i] = fields_data[field_name[i]]['stokesu']['ratio']['c']
    ratios_u[3,i] = fields_data[field_name[i]]['stokesu']['ratio']['d']
    

In [ ]:
print(np.nanmax(ratios_u[0]))

In [ ]:
fig, axs = plt.subplots(2,2,figsize=(13,10))

fs = 24
plt.subplots_adjust(left=0.09, bottom=0.1, right=0.98, top=0.98, wspace=0.13, hspace=0.13)

panels = [['(a)','(b)'],['(c)','(d)']]

min = -0.125
max = 100.125
bins = 402

axs[0,0].hist(ratios_q[0], bins=bins, range=(min,max),alpha=0.6, label='Stokes Q, band A',color='steelblue');
axs[0,0].hist(ratios_u[0], bins=bins, range=(min,max),alpha=0.6, label='Stokes U, band A',color='coral');

axs[0,1].hist(ratios_q[1], bins=bins, range=(min,max),alpha=0.6, label='Stokes Q, band B',color='steelblue');
axs[0,1].hist(ratios_u[1], bins=bins, range=(min,max),alpha=0.6, label='Stokes U, band B',color='coral');

axs[1,0].hist(ratios_q[2], bins=bins, range=(min,max),alpha=0.6, label='Stokes Q, band C',color='steelblue');
axs[1,0].hist(ratios_u[2], bins=bins, range=(min,max),alpha=0.6, label='Stokes U, band C',color='coral');

axs[1,1].hist(ratios_q[3], bins=bins, range=(min,max),alpha=0.6, label='Stokes Q, band D',color='steelblue');
axs[1,1].hist(ratios_u[3], bins=bins, range=(min,max),alpha=0.6, label='Stokes U, band D',color='coral');

for i in range(0,2):
    for j in range(0,2):
        axs[i,j].set_xlim(0,10)
        axs[i,j].set_ylim(0,60)
        #axs[i,j].grid()
        axs[i,j].legend(fontsize=fs)
        axs[i,j].tick_params(axis='both', labelsize=fs, 
                             left=True, right=True, bottom=True, top=True,which='both', 
                             width=2, length=6)
        axs[i,j].set_xticks([0,1,2,3,4,5,6,7,8,9,10])
        axs[i,j].text(0.75,55, panels[i][j], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center')
        for spine in axs[i,j].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
    
    axs[i,0].set_ylabel('Number of fields',fontsize=fs)
axs[1,0].set_xlabel('single-antenna/aperture-synthesis',fontsize=fs)
axs[1,1].set_xlabel('single-antenna/aperture-synthesis',fontsize=fs)

plt.savefig('../plots/uv_plane_ratio_allcgps_v2.pdf')

In [ ]:
chan = 0

thresh = 10

wbad = np.where((ratios_q[0,:] > thresh) | 
                (ratios_u[0,:] > thresh) |
                (ratios_q[1,:] > thresh) |
                (ratios_u[1,:] > thresh) |
                (ratios_q[2,:] > thresh) |
                (ratios_u[2,:] > thresh) |
                (ratios_q[3,:] > thresh) |
                (ratios_u[3,:] > thresh))[0]

print(len(field_name))
print(len(wbad))

for i in range(0,len(wbad)):
    print(field_name[wbad[i]],round(ratios_q[chan,wbad[i]],1),round(ratios_u[chan,wbad[i]],1))

In [ ]:
field_list_ann = []
field_lon_ann = []
field_lat_ann = []

i = 0
with open('../st-fields-mosaics/c21fieldsall.ann', 'r') as file:
    for line in file:
        if (i > 3):
            if line.strip().split()[0] == 'CIRCLE':
                field_list_ann.append(line.strip().split()[6].lower())
                field_lon_ann.append(float(line.strip().split()[2]))
                field_lat_ann.append(float(line.strip().split()[3]))
        i = i+1

field_lon_ann = np.array(field_lon_ann)
field_lat_ann = np.array(field_lat_ann)

plt.scatter(field_lon_ann,field_lat_ann)
print(np.nanmin(field_lon_ann),np.nanmax(field_lon_ann))

In [ ]:
chan = 0

thresh = 10

wbad = np.where((ratios_q[0,:] > thresh) | 
                (ratios_u[0,:] > thresh) |
                (ratios_q[1,:] > thresh) |
                (ratios_u[1,:] > thresh) |
                (ratios_q[2,:] > thresh) |
                (ratios_u[2,:] > thresh) |
                (ratios_q[3,:] > thresh) |
                (ratios_u[3,:] > thresh))[0]

print(len(field_name))
print(len(wbad))
print('')

bad_fields_lon = []
bad_fields_lat = []

for i in range(0,len(wbad)):
    print(field_name[wbad[i]],round(ratios_q[chan,wbad[i]],1),round(ratios_u[chan,wbad[i]],1))
    try:
        w_ann = field_list_ann.index(field_name[wbad[i]])
        print(field_list_ann[w_ann], field_lon_ann[w_ann], field_lat_ann[w_ann])
        bad_fields_lon.append(field_lon_ann[w_ann])
        bad_fields_lat.append(field_lat_ann[w_ann])
    except:
        pass
    print('')

bad_fields_lon = np.array(bad_fields_lon)
bad_fields_lat = np.array(bad_fields_lat)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(18,6))

ax.scatter(field_lon_ann,field_lat_ann)
ax.scatter(bad_fields_lon,bad_fields_lat)

ax.set_xlim(180,160)
ax.set_ylim(-5,6)
ax.grid()

Outliers:
1. artifact at 180
2. high lat near 160 - high PI
3. W3 near 134
4. low lat near 123 - high PI
5. hook near 120 - high PI
6. Cas A surroundings
7. Cyg A

In [ ]:
def make_radial_plots(data,field,stokes,band):
    
    fs=28
    lw = 2
    s = 60

    axs = ['ax1','ax2','ax3']
    panels = ['(a)','(b)','(c)']

    fig = plt.figure(figsize=(17,16))
    plt.subplots_adjust(top = 0.98, bottom = 0.06, right = 0.98, left = 0.08, hspace=0.11)

    axs[0] = fig.add_subplot(311)
    axs[0].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['raw'][band],color='C0',
                   label='DRAO ST',s=s,marker="o")
    axs[0].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['raw'][band],color='C4',
                   label='GMIMS-HBN',s=s,marker="s")
    handles,labels = axs[0].get_legend_handles_labels()
    order = [2,0,3,1]
    axs[0].set_ylim(1,5000)

    axs[1] = fig.add_subplot(312)
    axs[1].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['raw'][band],color='C0',
                   label='DRAO ST',s=s,marker="o")
    axs[1].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['deconv'][band],color='C4',
                   label='GMIMS-HBN: deconvolved, tapered',s=s,marker="s")
    axs[1].set_ylim(1,5000)
    handles,labels = axs[1].get_legend_handles_labels()
    order = [2,0,3,1]

    axs[2] = fig.add_subplot(313)
    axs[2].set_xlabel('Baseline (m)',fontsize=fs)
    axs[2].scatter(data[field]['uvr1d'][band],data[field][stokes]['cgps']['feather'][band],color='C0',
                   label='DRAO ST feathered',s=s,marker="o")
    axs[2].scatter(data[field]['uvr1d'][band],data[field][stokes]['gmims']['feather'][band],color='C4',
                   label='GMIMS-HBN feathered',s=s,marker="s")
    axs[2].set_ylim(1,5000)
    handles,labels = axs[2].get_legend_handles_labels()
    order = [2,0,3,1]

    for i in range(0,3):
        axs[i].tick_params(axis="x", labelsize=fs)
        axs[i].tick_params(axis="y", labelsize=fs)
        axs[i].set_xlim(0,50)
        axs[i].set_ylabel('$uv$-plane amplitude',fontsize=fs)
        #axs[i].plot([12.858,12.858],[0,80000],color='black',linestyle='dashed')
        #axs[i].plot([17.144,17.144,],[0,80000],color='black',linestyle='dashed')
        #axs[i].axvline(x=12.858, color='black',linestyle='dashed')
        axs[i].axvline(x=8.572, color='black',linestyle='dashed')
        axs[i].axvline(x=17.144,color='black',linestyle='dashed')
        axs[i].grid()
        axs[i].set_xticks([0,5,10,15,20,25,30,35,40,45,50])
        #axs[i].set_ticklabels(fontsize=fs)
        axs[i].set_yscale('log')
        axs[i].legend(fontsize=fs)
        axs[i].text(1.5,2500, panels[i], fontsize=fs+2, bbox={'facecolor':'white', 'alpha':1, 'edgecolor':'k'}, 
                          ha='center', va='center')
        axs[i].tick_params(axis='both', labelsize=fs, 
                             left=True, right=True, bottom=True,which='both', 
                             width=2, length=6)
        for spine in axs[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)

    plt.savefig('../plots/CGPS_GMIMS_UV_radial_new_v2.pdf')
    
    
    return

In [ ]:
make_radial_plots(fields_data,'ob12','stokesu','c')

In [ ]:
print(data['ob03']['uvr']['a'])
print(data['ob03']['uvr']['b'])
print(data['ob03']['uvr']['c'])
print(data['ob03']['uvr']['d'])

In [ ]:
field = 'ob03'
stokes = 'stokesq'
chan = 'a'

#print(data[field][stokes]['gmims']['raw'][chan])
#print(data[field][stokes]['gmims']['feather'][chan])
#print(data[field][stokes]['gmims']['deconv'][chan])
#print('')
#print(data[field][stokes]['cgps']['raw'][chan])
#print(data[field][stokes]['cgps']['feather'][chan])

In [ ]:
this = fits.open('/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/G_matched_uv_ob03_au.fits')
that = fits.open('/home2/DATA/CGPS_GMIMS_PhD/CGPS_GMIMS_PhD/THESIS_pics/G_feather_uv_ob03_au.fits')

In [ ]:
#plt.imshow(this[0].data[0,0,511-20:511+20,0:20]-that[0].data[0,0,511-20:511+20,0:20],vmin=-0.1,vmax=0.1)
plt.imshow(this[0].data[0,0,511-20:511+20,0:20],vmin=0,vmax=400)

In [ ]:
plt.imshow(that[0].data[0,0,511-20:511+20,0:20],vmin=0,vmax=400)

In [ ]:
C_raw_all_a = []
for field in fieldnames:
    C_raw_all_a.append(data[field]['stokesu']['cgps']['raw']['a'])

In [ ]:
print(C_raw_all_a)

In [ ]:
data['ob12']['stokesq']['cgps']['raw']['a']